<div dir="rtl">

# 🌐 05 - End-to-End Complete Data Ingestion Pipeline (من الصفر للاحتراف)

## ما هو استيعاب واستيراد البيانات (Data Ingestion)؟
- الخطوة الأولى والأساسية في أي نظام ذكاء اصطناعي أو **RAG Pipeline**.
- تهدف إلى سحب وتجميع البيانات غير المنظمة من مصادر متعددة ومتباينة (ملفات نصية، مستندات PDF، مواقع إنترنت، أوراق بحثية من arXiv، وبيانات هيكلية JSON) وتحويلها إلى كائنات موحدة تُسمى **`Document`**.

---

### 🎯 ما سنتعلمه ونطبقه في هذا الكراس:
1. **قراءة الملفات النصية الخام** عبر `TextLoader`.
2. **استخراج ومعالجة ملفات الـ PDF** ومقارنة `PyPDFLoader` و `PyMuPDFLoader`.
3. **سحب المقالات من الويب وتنظيفها** عبر `WebBaseLoader` و `BeautifulSoup`.
4. **جلب الأبحاث الأكاديمية تلقائياً** عبر `ArxivLoader`.
5. **توحيد وتنظيم البيانات الوصفية (Metadata Enrichment & Normalization)**.
6. **فحص وتصفية المستندات** والتأكد من جاهزيتها للمراحل القادمة (Splitting & Embedding).

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة وتحديد مسارات البيانات

</div>


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

# تحديد مسار مجلد البيانات بدقة
BASE_DIR = Path.cwd()
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")

print(f"📁 مسار مجلد البيانات: {DATA_DIR.resolve()}")
print(f"الملفات الموجودة: {[f.name for f in DATA_DIR.glob('*') if f.is_file()]}")


<div dir="rtl">

### 2️⃣ تحميل الملفات النصية (TextLoader) مع الحفاظ على ترميز UTF-8
قراءة النصوص الخام مع إضافة ميتاداتا مخصصة مثل نوع المصدر وعدد الأحرف.

</div>


In [ ]:
from langchain_community.document_loaders import TextLoader

text_file = DATA_DIR / "sample.txt"
text_loader = TextLoader(str(text_file), encoding="utf-8")
text_docs = text_loader.load()

# إثراء الميتاداتا
for doc in text_docs:
    doc.metadata["source_type"] = "plain_text"
    doc.metadata["char_count"] = len(doc.page_content)

print(f"✅ تم تحميل المستند النصي بنجاح! عدد الصفحات: {len(text_docs)}")
print(f"مقتطف من المحتوى: {text_docs[0].page_content[:150]}...")
print(f"الميتاداتا: {text_docs[0].metadata}")


<div dir="rtl">

### 3️⃣ استخراج نصوص الـ PDF (PyPDFLoader vs PyMuPDFLoader)
استخراج الصفحات وحساب معلومات كل صفحة على حدة.

</div>


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

pdf_file = DATA_DIR / "sample.pdf"

# استخدام PyMuPDFLoader لسرعته العالية ودقته في استخراج النصوص
pdf_loader = PyMuPDFLoader(str(pdf_file))
pdf_docs = pdf_loader.load()

for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"
    doc.metadata["char_count"] = len(doc.page_content)

print(f"✅ تم تحميل ملف الـ PDF بنجاح! عدد الصفحات: {len(pdf_docs)}")
if pdf_docs:
    print(f"صفحة 1: {pdf_docs[0].page_content.strip()[:150]}...")
    print(f"ميتاداتا: {pdf_docs[0].metadata}")


<div dir="rtl">

### 4️⃣ جلب صفحات الويب وتنقيتها (WebBaseLoader)
سحب مقال أو توثيق من الإنترنت مع تنظيف وسوم الـ HTML غير المرغوبة.

</div>


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

web_url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
web_loader = WebBaseLoader(
    web_paths=[web_url],
    # يمكن تحديد عناصر CSS معينة لسحبها فقط
)
web_docs = web_loader.load()

for doc in web_docs:
    doc.metadata["source_type"] = "web_page"
    doc.metadata["char_count"] = len(doc.page_content)

print(f"✅ تم جلب محتوى صفحة الويب بنجاح! الطول الإجمالي: {len(web_docs[0].page_content)} حرف.")
print(f"العنوان والمصدر: {web_docs[0].metadata.get('title', 'N/A')} - {web_docs[0].metadata.get('source')}")


<div dir="rtl">

### 5️⃣ البحث في الأوراق العلمية واستيرادها (ArxivLoader)
البحث عن أحدث الأوراق البحثية المتعلقة بـ RAG أو LLMs وجلب ملخصاتها وبياناتها الوصفية.

</div>


In [ ]:
from langchain_community.document_loaders import ArxivLoader

# استعلام بحث مباشر عن أوراق الـ RAG
arxiv_loader = ArxivLoader(query="Retrieval-Augmented Generation LLM", load_max_docs=2)
arxiv_docs = arxiv_loader.load()

for doc in arxiv_docs:
    doc.metadata["source_type"] = "arxiv_paper"
    doc.metadata["char_count"] = len(doc.page_content)

print(f"✅ تم جلب {len(arxiv_docs)} ورقة بحثية من arXiv بنجاح!")
for i, paper in enumerate(arxiv_docs, 1):
    print(f"{i}. العنوان: {paper.metadata.get('Title', 'N/A')}")
    print(f"   المؤلفون: {paper.metadata.get('Authors', 'N/A')}")
    print(f"   تاريخ النشر: {paper.metadata.get('Published', 'N/A')}\n")


<div dir="rtl">

### 6️⃣ توحيد وفحص كامل قاعدة المستندات (Corpus Consolidation & Validation)
تجميع المستندات من كافة المصادر المختلفة في قائمة موحدة وفحص جودتها قبل الانتقال لمرحلة التقسيم (Splitting).

</div>


In [ ]:
# تجميع كافة المستندات في قائمة واحدة موحدة
all_documents = []
all_documents.extend(text_docs)
all_documents.extend(pdf_docs)
all_documents.extend(web_docs)
all_documents.extend(arxiv_docs)

print("="*60)
print(f"🎉 إجمالي المستندات المجمعة: {len(all_documents)} مستند")
print("="*60)

# تقرير إحصائي حسب نوع المصدر
source_counts = {}
total_chars = 0

for doc in all_documents:
    st = doc.metadata.get("source_type", "unknown")
    source_counts[st] = source_counts.get(st, 0) + 1
    total_chars += len(doc.page_content)

print("📊 تفاصيل مصادر البيانات:")
for src, count in source_counts.items():
    print(f"  • {src}: {count} مستند")
print(f"\n📈 إجمالي عدد الأحرف في المجموعة: {total_chars:,} حرف.")
print("✅ جميع المستندات جاهزة الآن للانتقال إلى مرحلة Text Splitting!")
